In [1]:
# ============================================================
# SIMPLE GAP FILLING:
# Time interpolation + month/hour median fallback
# ============================================================

import pandas as pd
import numpy as np

In [ ]:
# ============================================================
# 0. SETTINGS
# ============================================================

INPUT_FILE = "fixed_and_profiler_combined_dataset.xlsx"
OUTPUT_FILE = "simple_gap_filled_dataset.xlsx"
GAP_REPORT_FILE = "simple_gap_report.csv"

DATETIME_COL = "datetime"
FREQ = "3h"

FILL_S_DIRECTLY = False

temp_cols = [
    "Water_Temp_0.5",
    "Water_Temp_1.0",
    "Water_Temp_2.5",
    "Water_Temp_4.5",
    "Water_Temp_6.5",
    "Water_Temp_8.5",
    "Water_Temp_10.5",
    "Water_Temp_12.5",
    "Water_Temp_14.5",
    "Water_Temp_16.5",
    "Water_Temp_18.5",
    "Water_Temp_20.5",
]

target_cols = temp_cols

In [3]:
# ============================================================
# 1. LOAD DATA
# ============================================================

df_raw = pd.read_excel(INPUT_FILE)

df_raw[DATETIME_COL] = pd.to_datetime(df_raw[DATETIME_COL], errors="coerce")
df_raw = df_raw.sort_values(DATETIME_COL).reset_index(drop=True)

print("Loaded dataset shape:", df_raw.shape)
print("Date range:", df_raw[DATETIME_COL].min(), "to", df_raw[DATETIME_COL].max())

Loaded dataset shape: (52518, 14)
Date range: 2007-07-13 12:00:00 to 2026-03-12 09:00:00


In [4]:
# ============================================================
# 2. CHECK EXPECTED COLUMNS
# ============================================================

needed_cols = [DATETIME_COL] + target_cols
missing_cols = [c for c in needed_cols if c not in df_raw.columns]

if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

print("\nColumns to fill:")
for col in target_cols:
    print("  -", col)


Columns to fill:
  - Water_Temp_0.5
  - Water_Temp_1.0
  - Water_Temp_2.5
  - Water_Temp_4.5
  - Water_Temp_6.5
  - Water_Temp_8.5
  - Water_Temp_10.5
  - Water_Temp_12.5
  - Water_Temp_14.5
  - Water_Temp_16.5
  - Water_Temp_18.5
  - Water_Temp_20.5


In [5]:
# ============================================================
# 3. FIX ONLY OFF-GRID TIMESTAMPS
# ============================================================
# Some timestamps may be at 11:00, 07:00, etc.
# This rounds only those few off-grid rows to the nearest 3-hour time.

full_datetime_index_check = pd.date_range(
    start=df_raw[DATETIME_COL].min(),
    end=df_raw[DATETIME_COL].max(),
    freq=FREQ
)

off_grid_mask = ~df_raw[DATETIME_COL].isin(full_datetime_index_check)

print("\nOriginal rows not matching strict 3-hour grid:", off_grid_mask.sum())

if off_grid_mask.sum() > 0:
    print("\nOff-grid timestamps before fixing:")
    print(df_raw.loc[off_grid_mask, [DATETIME_COL]])

    df_raw.loc[off_grid_mask, DATETIME_COL] = (
        df_raw.loc[off_grid_mask, DATETIME_COL].dt.round(FREQ)
    )

    df_raw = df_raw.sort_values(DATETIME_COL).reset_index(drop=True)

    print("\nOff-grid timestamps after rounding:")
    print(df_raw.loc[off_grid_mask, [DATETIME_COL]])


Original rows not matching strict 3-hour grid: 10

Off-grid timestamps before fixing:
                 datetime
42437 2022-02-21 11:00:00
43011 2022-05-04 07:00:00
43835 2022-08-16 13:00:00
43837 2022-08-16 19:00:00
43838 2022-08-16 22:00:00
45328 2023-05-24 11:00:00
46524 2023-10-26 07:00:00
47842 2024-05-24 16:00:00
50164 2025-05-08 19:00:00
50166 2025-05-09 01:00:00

Off-grid timestamps after rounding:
                 datetime
42437 2022-02-21 12:00:00
43011 2022-05-04 06:00:00
43835 2022-08-16 12:00:00
43837 2022-08-16 18:00:00
43838 2022-08-16 21:00:00
45328 2023-05-24 12:00:00
46524 2023-10-26 06:00:00
47842 2024-05-24 15:00:00
50164 2025-05-08 18:00:00
50166 2025-05-09 00:00:00


In [6]:
# ============================================================
# 4. CHECK DUPLICATE TIMESTAMPS AFTER ROUNDING
# ============================================================

n_duplicates = df_raw[DATETIME_COL].duplicated().sum()
print("\nDuplicate datetime rows after fixing off-grid times:", n_duplicates)

if n_duplicates > 0:
    print("Combining duplicate timestamps by averaging target columns.")

    df_raw = (
        df_raw
        .groupby(DATETIME_COL, as_index=False)[target_cols]
        .mean()
        .sort_values(DATETIME_COL)
        .reset_index(drop=True)
    )
else:
    print("No duplicate timestamps found.")


Duplicate datetime rows after fixing off-grid times: 0
No duplicate timestamps found.


In [7]:
# ============================================================
# 5. DETECT TIME GAPS
# ============================================================

expected_freq = pd.Timedelta(FREQ)

gap_check = df_raw[[DATETIME_COL]].copy()
gap_check["previous_datetime"] = gap_check[DATETIME_COL].shift(1)
gap_check["time_diff"] = gap_check[DATETIME_COL] - gap_check["previous_datetime"]

gap_report = gap_check[gap_check["time_diff"] > expected_freq].copy()

gap_report["missing_rows_inside_gap"] = (
    gap_report["time_diff"] / expected_freq
).astype(int) - 1

gap_report = gap_report.rename(columns={DATETIME_COL: "next_datetime"})
gap_report = gap_report[
    ["previous_datetime", "next_datetime", "time_diff", "missing_rows_inside_gap"]
]

gap_report.to_csv(GAP_REPORT_FILE, index=False)

print("\nNumber of detected time gaps:", len(gap_report))
print("Missing 3-hour rows from time gaps:", gap_report["missing_rows_inside_gap"].sum())
print("Gap report saved as:", GAP_REPORT_FILE)

if len(gap_report) > 0:
    print("\nLargest gaps:")
    print(gap_report.sort_values("time_diff", ascending=False).head(10))


Number of detected time gaps: 64
Missing 3-hour rows from time gaps: 2018
Gap report saved as: simple_gap_report.csv

Largest gaps:
        previous_datetime       next_datetime        time_diff  \
45328 2023-04-14 03:00:00 2023-05-24 12:00:00 40 days 09:00:00   
42437 2022-01-20 00:00:00 2022-02-21 12:00:00 32 days 12:00:00   
44603 2022-12-16 18:00:00 2023-01-13 12:00:00 27 days 18:00:00   
48028 2024-07-20 12:00:00 2024-08-09 15:00:00 20 days 03:00:00   
44602 2022-11-28 06:00:00 2022-12-16 18:00:00 18 days 12:00:00   
47825 2024-05-02 06:00:00 2024-05-17 21:00:00 15 days 15:00:00   
47996 2024-07-06 09:00:00 2024-07-16 15:00:00 10 days 06:00:00   
47854 2024-05-28 00:00:00 2024-06-05 12:00:00  8 days 12:00:00   
47983 2024-06-26 03:00:00 2024-07-04 12:00:00  8 days 09:00:00   
47215 2024-02-02 09:00:00 2024-02-09 12:00:00  7 days 03:00:00   

       missing_rows_inside_gap  
45328                      322  
42437                      259  
44603                      221  
48028   

In [8]:
# ============================================================
# 6. CREATE COMPLETE 3-HOURLY TIMELINE
# ============================================================

full_datetime_index = pd.date_range(
    start=df_raw[DATETIME_COL].min(),
    end=df_raw[DATETIME_COL].max(),
    freq=FREQ
)

original_datetimes = set(df_raw[DATETIME_COL])

df = (
    df_raw
    .set_index(DATETIME_COL)
    .reindex(full_datetime_index)
)

df.index.name = DATETIME_COL

df["row_created_from_time_gap"] = ~df.index.isin(original_datetimes)

print("\nRows before creating time-gap rows:", len(df_raw))
print("Rows after creating complete 3-hourly timeline:", len(df))
print("New rows created from time gaps:", df["row_created_from_time_gap"].sum())


Rows before creating time-gap rows: 52518
Rows after creating complete 3-hourly timeline: 54536
New rows created from time gaps: 2018


In [9]:
# ============================================================
# 7. FLAG ORIGINALLY MISSING VALUES
# ============================================================

for col in target_cols:
    df[col + "_was_missing"] = df[col].isna().astype(int)

print("\nMissing values before filling:")
print(df[target_cols].isna().sum())
print("\nTotal missing cells before filling:", df[target_cols].isna().sum().sum())


Missing values before filling:
Water_Temp_0.5      9635
Water_Temp_1.0      5273
Water_Temp_2.5      9754
Water_Temp_4.5      9757
Water_Temp_6.5      9758
Water_Temp_8.5      9756
Water_Temp_10.5     9770
Water_Temp_12.5    16496
Water_Temp_14.5     9758
Water_Temp_16.5     9762
Water_Temp_18.5    16437
Water_Temp_20.5     9773
dtype: int64

Total missing cells before filling: 125929


In [10]:
# ============================================================
# 8. FILL USING TIME INTERPOLATION
# ============================================================
# This fills values based on nearby values in time.

df_filled = df.copy()

df_filled[target_cols] = df_filled[target_cols].interpolate(
    method="time",
    limit_direction="both"
)

print("\nMissing values after time interpolation:")
print(df_filled[target_cols].isna().sum())
print("\nTotal missing cells after time interpolation:", df_filled[target_cols].isna().sum().sum())


Missing values after time interpolation:
Water_Temp_0.5     0
Water_Temp_1.0     0
Water_Temp_2.5     0
Water_Temp_4.5     0
Water_Temp_6.5     0
Water_Temp_8.5     0
Water_Temp_10.5    0
Water_Temp_12.5    0
Water_Temp_14.5    0
Water_Temp_16.5    0
Water_Temp_18.5    0
Water_Temp_20.5    0
dtype: int64

Total missing cells after time interpolation: 0


In [ ]:
# ============================================================
# 9. FILL REMAINING VALUES USING MONTH-HOUR MEDIAN
# ============================================================
# If anything is still missing, this section will let you use the typical value from the same month and hour. For my case, there was no remaining missing value after interpolation, so this step is just a structure for future datasets with more missing values.

df_filled["month"] = df_filled.index.month
df_filled["hour"] = df_filled.index.hour

for col in target_cols:
    if df_filled[col].isna().any():
        df_filled[col] = df_filled[col].fillna(
            df_filled.groupby(["month", "hour"])[col].transform("median")
        )

    if df_filled[col].isna().any():
        df_filled[col] = df_filled[col].fillna(df_filled[col].median())

df_filled = df_filled.drop(columns=["month", "hour"])

print("\nMissing values after month-hour median fallback:")
print(df_filled[target_cols].isna().sum())
print("\nTotal missing cells after all filling:", df_filled[target_cols].isna().sum().sum())


Missing values after month-hour median fallback:
Water_Temp_0.5     0
Water_Temp_1.0     0
Water_Temp_2.5     0
Water_Temp_4.5     0
Water_Temp_6.5     0
Water_Temp_8.5     0
Water_Temp_10.5    0
Water_Temp_12.5    0
Water_Temp_14.5    0
Water_Temp_16.5    0
Water_Temp_18.5    0
Water_Temp_20.5    0
dtype: int64

Total missing cells after all filling: 0


In [12]:
# ============================================================
# 10. CHECK THAT OBSERVED VALUES WERE NOT CHANGED
# ============================================================

original_indexed = df_raw.set_index(DATETIME_COL).sort_index()

changed_count = 0

for col in target_cols:
    observed_mask = original_indexed[col].notna()

    original_values = original_indexed.loc[observed_mask, col]
    filled_values = df_filled.loc[original_values.index, col]

    changed = ~np.isclose(
        original_values.values,
        filled_values.values,
        equal_nan=True
    )

    changed_count += changed.sum()

print("\nObserved values changed during filling:", changed_count)

if changed_count == 0:
    print("Good: original observed values were preserved.")
else:
    print("Warning: some observed values changed. Check the filling process.")


Observed values changed during filling: 0
Good: original observed values were preserved.


In [13]:
# ============================================================
# 11. OPTIONAL BASIC PHYSICAL CHECK
# ============================================================

print("\nFilled value ranges:")

for col in target_cols:
    print(
        col,
        "min =", round(df_filled[col].min(), 3),
        "max =", round(df_filled[col].max(), 3)
    )


# ============================================================
# 12. SAVE OUTPUT
# ============================================================

output = df_filled.reset_index()

ordered_cols = (
    [DATETIME_COL]
    + target_cols
    + ["row_created_from_time_gap"]
    + [col + "_was_missing" for col in target_cols]
)

output = output[ordered_cols]

output.to_excel(OUTPUT_FILE, index=False)

print("\nSaved simple gap-filled dataset as:", OUTPUT_FILE)


Filled value ranges:
Water_Temp_0.5 min = 7.21 max = 24.999
Water_Temp_1.0 min = 7.14 max = 26.7
Water_Temp_2.5 min = 7.25 max = 25.02
Water_Temp_4.5 min = 7.19 max = 25.72
Water_Temp_6.5 min = 7.5 max = 25.47
Water_Temp_8.5 min = 7.28 max = 24.54
Water_Temp_10.5 min = 7.36 max = 24.14
Water_Temp_12.5 min = 7.54 max = 25.19
Water_Temp_14.5 min = 7.34 max = 25.32
Water_Temp_16.5 min = 7.4 max = 24.49
Water_Temp_18.5 min = 7.24 max = 27.47
Water_Temp_20.5 min = 7.16 max = 28.23

Saved simple gap-filled dataset as: simple_gap_filled_dataset.xlsx
